In [11]:
import pandas as pd

df = pd.read_csv('/data2/yuyao/methane_emission/carbon_mapper_data/csvs/raw_s2_90360_cleaned.csv')

# df['date'] = pd.to_datetime(df['s2_datetime'], format='ISO8601', utc=True)

train_df = df[(df['datetime'] < '2020-07-08') | (df['datetime'] > '2021-08-10')]
test_df  = df[(df['datetime'] >= '2020-07-08') & (df['datetime'] <= '2021-08-10')]

print(len(train_df))
print(len(test_df))

3444
779


In [5]:
from pathlib import Path
import shutil

s2_root = Path("../carbonmapper_data_s2_l2a_reclip")
mask_root = Path("../carbon_mapper_data_masks")

for s2_dir in s2_root.iterdir():
    if not s2_dir.is_dir():
        continue

    mask_dir = mask_root / s2_dir.name
    if not mask_dir.is_dir():
        print(f"missing mask dir: {mask_dir}")
        continue

    candidates = sorted(s2_dir.glob("s2_*.tif"))
    if not candidates:
        print(f"no s2 tif in: {s2_dir}")
        continue

    src = candidates[0]
    dst = mask_dir / "s2.tif"

    if dst.exists():
        print(f"skip (exists): {dst}")
        continue

    shutil.copy2(src, dst)  # Translated comment
    print(f"copied: {src} -> {dst}")

copied: ../carbonmapper_data_s2_l2a_reclip/tan20250525t060542c29s4001-Z/s2_20250525T051711Z.tif -> ../carbon_mapper_data_masks/tan20250525t060542c29s4001-Z/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/tan20250518t074002c00s4001-J/s2_20250518T070641Z.tif -> ../carbon_mapper_data_masks/tan20250518t074002c00s4001-J/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/GAO20240508t181643p0000-D/s2_20240509T172901Z.tif -> ../carbon_mapper_data_masks/GAO20240508t181643p0000-D/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/GAO20230825t183530p0000-A/s2_20230825T171859Z.tif -> ../carbon_mapper_data_masks/GAO20230825t183530p0000-A/s2.tif


copied: ../carbonmapper_data_s2_l2a_reclip/GAO20230906t180335p0000-C/s2_20230906T175919Z.tif -> ../carbon_mapper_data_masks/GAO20230906t180335p0000-C/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/ang20220502t170537-A/s2_20220501T174859Z.tif -> ../carbon_mapper_data_masks/ang20220502t170537-A/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/GAO20240508t165509p0000-E/s2_20240507T173909Z.tif -> ../carbon_mapper_data_masks/GAO20240508t165509p0000-E/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/GAO20230912t193133p0000-B/s2_20230913T174909Z.tif -> ../carbon_mapper_data_masks/GAO20230912t193133p0000-B/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/ang20220430t172719-A/s2_20220429T175921Z.tif -> ../carbon_mapper_data_masks/ang20220430t172719-A/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/GAO20240611t205814p0000-A/s2_20240611T182919Z.tif -> ../carbon_mapper_data_masks/GAO20240611t205814p0000-A/s2.tif
copied: ../carbonmapper_data_s2_l2a_reclip/ang20200828t214929-A/s2_20200829T18

In [1]:
import os
import shutil
from pathlib import Path

base_dir = Path("/data2/yuyao/methane_emission")
s2_root = base_dir / "carbon_mapper_data" / "CM_S2_L2A"
mask_root = base_dir / "carbon_mapper_data_masks"
gee_root = base_dir / "carbonmapper_data_s2_l2a_gee_download"
out_root = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360")

out_root.mkdir(parents=True, exist_ok=True)

# Translated comment
plume_ids = []
for p in s2_root.iterdir():
    if not p.is_dir():
        continue
    s2_path = p / "s2.tif"
    if s2_path.exists():
        plume_ids.append(p.name)

for plume_id in plume_ids:
    s2_path = s2_root / plume_id / "s2.tif"
    if not s2_path.exists():
        continue

    out_dir = out_root / plume_id
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) S2
    shutil.copy2(s2_path, out_dir / "s2.tif")

    # 2) masks
    mask_dir = mask_root / plume_id
    for name in ["plume.tif", "reprojected.tif", "resized_512x512.tif"]:
        src = mask_dir / name
        if src.exists():
            shutil.copy2(src, out_dir / name)

    # 3) GEE download files by prefix
    for f in gee_root.glob(f"{plume_id}_*.tif"):
        shutil.copy2(f, out_dir / f.name)

print(f"done, total folders: {len(plume_ids)}")


done, total folders: 4358


In [2]:
from pathlib import Path
import pandas as pd

root = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360")
meta_csv = Path("/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file.csv")

# Translated comment
meta = pd.read_csv(meta_csv, usecols=["plume_id", "datetime", "plume_latitude", "plume_longitude"])
meta_map = meta.set_index("plume_id").to_dict(orient="index")

rows = []
for plume_dir in sorted(p for p in root.iterdir() if p.is_dir()):
    plume_id = plume_dir.name

    def pick(name):
        p = plume_dir / name
        return str(p) if p.exists() else None

    # 90 -> reference_month, 360 -> reference_year
    month_path = pick(f"{plume_id}_reference_month.tif")
    year_path = pick(f"{plume_id}_reference_year.tif")

    # leak_0/1/2/3
    leak0 = pick(f"{plume_id}_leak_0.tif")
    leak1 = pick(f"{plume_id}_leak_1.tif")
    leak2 = pick(f"{plume_id}_leak_2.tif")
    leak3 = pick(f"{plume_id}_leak_3.tif")

    meta_row = meta_map.get(plume_id, {})

    rows.append({
        "plume_id": plume_id,
        "datetime": meta_row.get("datetime"),
        "plume_latitude": meta_row.get("plume_latitude"),
        "plume_longitude": meta_row.get("plume_longitude"),
        "s2_path": pick("s2.tif"),
        "s2_90_path": month_path,
        "s2_360_path": year_path,
        "plume_path": pick("plume.tif"),
        "reprojected_path": pick("reprojected.tif"),
        "resized_512x512_path": pick("resized_512x512.tif"),
        "leak_0_path": leak0,
        "leak_1_path": leak1,
        "leak_2_path": leak2,
        "leak_3_path": leak3,
    })

df = pd.DataFrame(rows)

out_csv = root / "plume_raw_s2_90360_index.csv"
df.to_csv(out_csv, index=False)

missing_total = int(df.isna().sum().sum())
missing_per_col = df.isna().sum().to_dict()
complete_rows = int(df.notna().all(axis=1).sum())

print(f"Wrote: {out_csv}")
print("Missing per column:", missing_per_col)
print("Total missing values:", missing_total)
print("Complete rows (no missing):", complete_rows)

Wrote: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360/plume_raw_s2_90360_index.csv
Missing per column: {'plume_id': 0, 'datetime': 0, 'plume_latitude': 0, 'plume_longitude': 0, 's2_path': 0, 's2_90_path': 28, 's2_360_path': 28, 'plume_path': 2, 'reprojected_path': 2, 'resized_512x512_path': 109, 'leak_0_path': 28, 'leak_1_path': 2308, 'leak_2_path': 4024, 'leak_3_path': 4092}
Total missing values: 10621
Complete rows (no missing): 250


In [4]:
import os
import pandas as pd
import tifffile as tiff
import numpy as np
import random

# =========================
# Config
# =========================
csv_path = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/raw_s2_90360_cleaned.csv"
cols = ["s2_path", "s2_90_path", "s2_360_path"]

band_index = 11  # Translated comment
zero_thresh = 0.2
sample_n_per_col = 10  # Translated comment
seed = 42

# Translated comment
save_png = False
png_out_dir = "./random_check_png"

# =========================
# Helpers
# =========================
def normalize_to_uint8(arr):
    """Robust normalize for visualization."""
    arr = arr.astype(np.float32)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr, dtype=np.uint8)
    vmin = np.percentile(arr[finite], 2)
    vmax = np.percentile(arr[finite], 98)
    if vmax <= vmin:
        return np.zeros_like(arr, dtype=np.uint8)
    x = (arr - vmin) / (vmax - vmin)
    x = np.clip(x, 0, 1)
    return (x * 255).astype(np.uint8)

def read_s2_cube_and_band(path, band_index=11):
    """
    Return:
      info dict:
        - shape
        - layout: 'BHW' or 'HWB'
        - W, H, bands
        - zero_ratio
      band_2d array (H,W) of chosen band
    """
    if path is None or (isinstance(path, float) and np.isnan(path)):
        return None, None, "NA path"

    if not os.path.exists(path):
        return None, None, f"NOT FOUND: {path}"

    try:
        with tiff.TiffFile(path) as tif:
            data = tif.asarray()
    except Exception as e:
        return None, None, f"READ ERROR: {e}"

    if data.ndim != 3:
        return None, None, f"UNEXPECTED ndim={data.ndim}, shape={getattr(data, 'shape', None)}"

    s0, s1, s2 = data.shape

    # Translated comment
    if s0 in (12, 13):
        layout = "BHW"
        bands, H, W = data.shape
        if band_index >= bands:
            return None, None, f"BAND INDEX OUT OF RANGE: bands={bands}, band_index={band_index}"
        band = data[band_index, :, :]
    elif s2 in (12, 13):
        layout = "HWB"
        H, W, bands = data.shape
        if band_index >= bands:
            return None, None, f"BAND INDEX OUT OF RANGE: bands={bands}, band_index={band_index}"
        band = data[:, :, band_index]
    else:
        return None, None, f"CAN'T INFER LAYOUT: shape={data.shape} (expect one dim in 12/13)"

    # zero ratio
    band_np = np.asarray(band)
    total = band_np.size
    zero_ratio = float(np.sum(band_np == 0)) / float(total) if total > 0 else np.nan

    info = {
        "shape": tuple(data.shape),
        "layout": layout,
        "bands": int(bands),
        "H": int(H),
        "W": int(W),
        "zero_ratio": float(zero_ratio),
    }
    return info, band_np, None

# =========================
# Main
# =========================
df = pd.read_csv(csv_path)

random.seed(seed)
np.random.seed(seed)

if save_png:
    os.makedirs(png_out_dir, exist_ok=True)
    try:
        import imageio.v2 as imageio
    except Exception:
 raise RuntimeError("save_png=True imageio: pip install imageio")

for col in cols:
    valid_paths = df[col].dropna().tolist()
    if len(valid_paths) == 0:
        print(f"\n[{col}] No paths found.")
        continue

    n = min(sample_n_per_col, len(valid_paths))
    sampled = random.sample(valid_paths, n)

    print(f"\n==================== Random check: {col} (n={n}) ====================")
    for i, p in enumerate(sampled, 1):
        info, band, err = read_s2_cube_and_band(p, band_index=band_index)
        if err is not None:
            print(f"[{i:02d}] ERROR: {err}")
            continue

        flag = "OK"
        if info["zero_ratio"] > zero_thresh:
            flag = f"ZERO>{int(zero_thresh*100)}%"

        print(
            f"[{i:02d}] {flag} | shape={info['shape']} layout={info['layout']} "
            f"W×H={info['W']}×{info['H']} bands={info['bands']} "
            f"zero_ratio={info['zero_ratio']:.4f}\n     {p}"
        )

        if save_png and band is not None:
            out_png = os.path.join(
                png_out_dir,
                f"{col}_{i:02d}_W{info['W']}_H{info['H']}_zr{info['zero_ratio']:.3f}.png"
            )
            imageio.imwrite(out_png, normalize_to_uint8(band))



==================== Random check: s2_path (n=10) ====================
[01] OK | shape=(12, 512, 512) layout=BHW W×H=512×512 bands=12 zero_ratio=0.0000
     /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360/GAO20210920t204107p0000-A/s2.tif
[02] OK | shape=(12, 512, 512) layout=BHW W×H=512×512 bands=12 zero_ratio=0.0000
     /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360/GAO20200714t172448p0000-E/s2.tif
[03] OK | shape=(12, 512, 512) layout=BHW W×H=512×512 bands=12 zero_ratio=0.0000
     /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360/ang20171016t175348-A/s2.tif
[04] OK | shape=(12, 512, 512) layout=BHW W×H=512×512 bands=12 zero_ratio=0.0000
     /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360/GAO20240611t192759p0000-F/s2.tif
[05] OK | shape=(12, 512, 512) layout=BHW W×H=512×512 bands=12 zero_ratio=0.0000
     /mnt/engg-leung/Research_No9_Methane_Emissions/Yuya

In [7]:
csv = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_raw_s2_90360/plume_raw_s2_90360_index.csv"
df = pd.read_csv(csv)
df = df.drop(columns=['leak_0_path', 'leak_1_path', 'leak_2_path', 'leak_3_path'])
missing_total = int(df.isna().sum().sum())
missing_per_col = df.isna().sum().to_dict()
complete_rows = int(df.notna().all(axis=1).sum())
df = df.dropna()
df.to_csv("/data2/yuyao/methane_emission/carbon_mapper_data/csvs/raw_s2_90360_cleaned.csv", index=False)

print("Missing per column:", missing_per_col)
print("Total missing values:", missing_total)
print("Complete rows (no missing):", complete_rows)

Missing per column: {'plume_id': 0, 'datetime': 0, 'plume_latitude': 0, 'plume_longitude': 0, 's2_path': 0, 's2_90_path': 28, 's2_360_path': 28, 'plume_path': 2, 'reprojected_path': 2, 'resized_512x512_path': 109}
Total missing values: 169
Complete rows (no missing): 4223


In [11]:
import pandas as pd

base_path = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file.csv"
s2_path = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file_with_s2_reclip.csv"
out_path = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file_joined_s2_reclip.csv"

base = pd.read_csv(base_path)
s2 = pd.read_csv(s2_path)

# Translated comment
s2 = s2[s2["tif_path"].notna() & (s2["tif_path"].astype(str).str.len() > 0)]

# Translated comment
merged = base.merge(s2, on="plume_id", how="inner")
merged = merged.rename(columns={"tif_path": "s2_path"})

merged.to_csv(out_path, index=False)
print(f"saved: {out_path}")

saved: /data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file_joined_s2_reclip.csv


In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import rasterio
import tifffile
import json
base_dir = '/data2/yuyao/methane_emission/carbon_mapper_data_masks'

csv_df = pd.read_csv('/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file_joined_s2_reclip.csv')

s2_files = []
cnt = 0
for dir in os.listdir(base_dir):
    # file_path = os.path.join(base_dir, dir, 's2_-5.tif')
    file_path2 = os.path.join(base_dir, dir, 's2.tif')
    file_path3 = os.path.join(base_dir, dir, 'resized_512x512.tif')
    # file_path4 = os.path.join(base_dir, dir, 's2_+5.tif')
    if os.path.exists(file_path2) and os.path.exists(file_path3):
        data = tifffile.imread(file_path2)
        total_elements = data[11].size
        zero_count = np.sum(data[11] == 0)
        if zero_count > (total_elements / 2): 
            continue

        # data = tifffile.imread(file_path)
        # total_elements = data[11].size
        # zero_count = np.sum(data[11] == 0)
        # if zero_count > (total_elements / 2): 
        #     continue

        # data = tifffile.imread(file_path4)
        # total_elements = data[11].size
        # zero_count = np.sum(data[11] == 0)
        # if zero_count > (total_elements / 2): 
        #     continue
        s2_files.append(dir)
        cnt += 1
print(f'total file count {cnt}')
data = []
for dir in s2_files:
    path = os.path.join(base_dir, dir)
    plume_path = os.path.join(base_dir, dir, 'resized_512x512.tif')
    s2_path = os.path.join(base_dir, dir, 's2.tif')
    # s2_pre_path = os.path.join(base_dir, dir, 's2_-5.tif')
    # s2_post_path = os.path.join(base_dir, dir, 's2_+5.tif')
    result = csv_df.loc[csv_df['plume_id'] == dir]
    data.append({"plume_id": dir, "plume_mask_path": plume_path, "s2_path": s2_path, "emission_auto": result['emission_auto'].values[0], "emission_uncertainty_auto": result['emission_uncertainty_auto'].values[0], "latitude": result['plume_latitude'].values[0], "longitude": result['plume_longitude'].values[0], "datetime": result['datetime'].values[0]})

df = pd.DataFrame(data)
print(df.columns)

df['date'] = pd.to_datetime(df['datetime'])
train_df = df[(df['date'] < '2020-07-08') | (df['date'] > '2023-08-10')] 
test_df = df[(df['date'] >= '2020-07-08') & (df['date'] <= '2023-08-10')]

train_csv = '/data2/yuyao/methane_emission/data_csv/train.csv'
test_csv = '/data2/yuyao/methane_emission/data_csv/test.csv'

# train_df = pd.DataFrame(train_data)
train_df.to_csv(train_csv, index=False)

# test_df = pd.DataFrame(test_data)
test_df.to_csv(test_csv, index=False)
print(cnt)
print(len(train_df))
print(len(test_df))


total file count 3402
Index(['plume_id', 'plume_mask_path', 's2_path', 'emission_auto',
       'emission_uncertainty_auto', 'latitude', 'longitude', 'datetime'],
      dtype='object')
3402
2920
482


In [13]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import rasterio
import tifffile
import json
import random

base_dir = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout_1/carbonmapper_data_temporal_split_classification'
train_csv_path = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout_1/carbonmapper_data_temporal_split_classification/train.csv'
test_csv_path = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout_1/carbonmapper_data_temporal_split_classification/test.csv'
origin_train_csv = '/data2/yuyao/methane_emission/data_csv/train.csv'
origin_test_csv = '/data2/yuyao/methane_emission/data_csv/test.csv'


def get_crop(width=512, height=512, center_size=30, crop_width=96, crop_height=96):
    """
 (top, left), (y, x), HWC: img[top:top+H, left:left+W, :]
    """
    center_x = width // 2
    center_y = height // 2
    center_left = center_x - center_size // 2
    center_right = center_x + center_size // 2
    center_top = center_y - center_size // 2
    center_bottom = center_y + center_size // 2

    # Translated comment
    left = random.randint(0, max(0, center_left - crop_width))
    top = random.randint(0, max(0, center_top - crop_height))

    if left + crop_width < center_right:
        left = center_right - crop_width
    if top + crop_height < center_bottom:
        top = center_bottom - crop_height
    return top, left


data = []

image_size = 96
max_left = 512 - image_size      # 416
quad_max = 256 - image_size      # 160

cnt = 0
org_train_df = pd.read_csv(origin_train_csv)

# Translated comment
for index, row in org_train_df.iterrows():
    # t_data: HWC = (H, W, C) = (512, 512, 12)
    t_data = tifffile.imread(row['s2_path'])
    mask = tifffile.imread(row['plume_mask_path'])

    crop_list = []
    # Translated comment
    for i in range(8):
        crop_list.append(get_crop())

    # Translated comment
    for i in range(12):
        # Translated comment
        crop_list.append((random.randint(0, max_left), random.randint(0, max_left)))

    for crop in crop_list:
        top, left = crop  # (y, x)

        nt_data = t_data[top:top + image_size, left:left + image_size, :]  # HWC
        n_mask = mask[top:top + image_size, left:left + image_size]

        dir_path = os.path.join(base_dir, str(cnt))
        os.makedirs(dir_path, exist_ok=True)

        nt_path = os.path.join(base_dir, str(cnt), "s2.tif")
        tifffile.imwrite(nt_path, nt_data)  # Translated comment

        n_mask_path = os.path.join(base_dir, str(cnt), "plume.tif")
        tifffile.imwrite(n_mask_path, n_mask)

        mask_sum = np.sum(n_mask)
        data.append({
            "id": cnt,
            "s2_path": nt_path,
            "plume_mask_path": n_mask_path,
            "label": 0 if mask_sum == 0 else 1,
            "emission_auto": 0 if mask_sum == 0 else row['emission_auto'],
            "emission_uncertainty_auto": 0 if mask_sum == 0 else row['emission_uncertainty_auto']
        })
        cnt += 1

train_df = pd.DataFrame(data)
train_df.to_csv(train_csv_path, index=False)
print(f'training set count: {cnt}')


# Translated comment
data_test = []
org_test_df = pd.read_csv(origin_test_csv)

for index, row in org_test_df.iterrows():
    t_data = tifffile.imread(row['s2_path'])        # HWC
    mask = tifffile.imread(row['plume_mask_path'])  # HW

    crop_list = []

    # Translated comment
    crop_list.append((random.randint(0, quad_max), random.randint(0, quad_max)))  # Translated comment
    crop_list.append((random.randint(256, max_left), random.randint(0, quad_max)))  # Translated comment
    crop_list.append((random.randint(0, quad_max), random.randint(256, max_left)))  # Translated comment
    crop_list.append((random.randint(256, max_left), random.randint(256, max_left)))  # Translated comment

    # Translated comment
    crop_list.append((256 - image_size // 2, 256 - image_size // 2))

    for crop in crop_list:
        top, left = crop

        nt_data = t_data[top:top + image_size, left:left + image_size, :]  # HWC
        n_mask = mask[top:top + image_size, left:left + image_size]

        dir_path = os.path.join(base_dir, str(cnt))
        os.makedirs(dir_path, exist_ok=True)

        nt_path = os.path.join(base_dir, str(cnt), "s2.tif")
        tifffile.imwrite(nt_path, nt_data)

        n_mask_path = os.path.join(base_dir, str(cnt), "plume.tif")
        tifffile.imwrite(n_mask_path, n_mask)

        mask_sum = np.sum(n_mask)
        data_test.append({
            "id": cnt,
            "s2_path": nt_path,
            "plume_mask_path": n_mask_path,
            "label": 0 if mask_sum == 0 else 1,
            "emission_auto": 0 if mask_sum == 0 else row['emission_auto'],
            "emission_uncertainty_auto": 0 if mask_sum == 0 else row['emission_uncertainty_auto']
        })
        cnt += 1

test_df = pd.DataFrame(data_test)
test_df.to_csv(test_csv_path, index=False)
print(f'total count: {cnt}')

column_sum = train_df['label'].sum()
print(f"label: {column_sum} total {len(train_df)}")

column_sum = test_df['label'].sum()
print(f"label: {column_sum} total {len(test_df)}")

In [2]:
import tifffile as tiff
arr = tiff.imread("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_90360_32/GAO20230825t180240p0000-A/s2.tif")
print(arr.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_90360_32/GAO20230825t180240p0000-A/s2.tif'

In [7]:
import pandas as pd

base_dir = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/plume_s2_90360_32_fixed_512"
train_path = f"{base_dir}/train.csv"
test_path = f"{base_dir}/test.csv"

def balance_1_1(df, label_col="label", seed=42):
    pos = df[df[label_col] == 1]
    neg = df[df[label_col] == 0]
    if len(neg) < len(pos):
        raise ValueError(f"neg<{label_col}=0>({len(neg)}) < pos({len(pos)}), can't downsample 0 to 1:1")
    neg = neg.sample(n=len(pos), random_state=seed)
    out = pd.concat([pos, neg], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# train_bal = balance_1_1(train_df)
# test_bal = balance_1_1(test_df)

# train_bal.to_csv(f"{base_dir}/train_balanced.csv", index=False)
# test_bal.to_csv(f"{base_dir}/test_balanced.csv", index=False)

# print("train:", len(train_bal), "label1:", train_bal["label"].sum())
# print("test :", len(test_bal), "label1:", test_bal["label"].sum())

print("train:", len(train_df), "label1:", train_df["label"].sum(), "ratio:", train_df["label"].sum()/len(train_df))
print("test :", len(test_df), "label1:", test_df["label"].sum(), "ratio:", test_df["label"].sum()/len(test_df))

train: 235453 label1: 118411 ratio: 0.5029071619389007
test : 58118 label1: 29106 ratio: 0.5008086995423104


In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import rasterio
import tifffile
import json
base_dir = '/home/hongxuan/methane/methane_emission/carbonmapper_data'

csv_df = pd.read_csv('/home/hongxuan/methane/methane_emission/carbonmapper/merged_file.csv')

s2_files = []
cnt = 0
for dir in os.listdir(base_dir):
    file_path = os.path.join(base_dir, dir, 's2_-5.tif')
    file_path2 = os.path.join(base_dir, dir, 's2.tif')
    file_path3 = os.path.join(base_dir, dir, 'resized_512x512.tif')
    file_path4 = os.path.join(base_dir, dir, 's2_+5.tif')
    if os.path.exists(file_path) and os.path.exists(file_path2) and os.path.exists(file_path3) and os.path.exists(file_path4):
        data = tifffile.imread(file_path2)
        total_elements = data[11].size
        zero_count = np.sum(data[11] == 0)
        if zero_count > (total_elements / 2): 
            continue

        data = tifffile.imread(file_path)
        total_elements = data[11].size
        zero_count = np.sum(data[11] == 0)
        if zero_count > (total_elements / 2): 
            continue

        data = tifffile.imread(file_path4)
        total_elements = data[11].size
        zero_count = np.sum(data[11] == 0)
        if zero_count > (total_elements / 2): 
            continue
        s2_files.append(dir)
        cnt += 1
print(f'total file count {cnt}')
data = []
for dir in s2_files:
    path = os.path.join(base_dir, dir)
    plume_path = os.path.join(base_dir, dir, 'resized_512x512.tif')
    s2_path = os.path.join(base_dir, dir, 's2.tif')
    s2_pre_path = os.path.join(base_dir, dir, 's2_-5.tif')
    s2_post_path = os.path.join(base_dir, dir, 's2_+5.tif')
    result = csv_df.loc[csv_df['plume_id'] == dir]
    data.append({"plume_id": dir, "plume_mask_path": plume_path, "s2_path": s2_path, "s2_pre_path": s2_pre_path, "s2_post_path": s2_post_path, "emission_auto": result['emission_auto'].values[0], "emission_uncertainty_auto": result['emission_uncertainty_auto'].values[0], "latitude": result['plume_latitude'].values[0], "longitude": result['plume_longitude'].values[0]})

train_csv = '/home/hongxuan/methane/methane_emission/carbonmapper_data/train.csv'
test_csv = '/home/hongxuan/methane/methane_emission/carbonmapper_data/test.csv'
test_size=0.2
train_data, test_data = train_test_split(data, test_size=test_size, random_state=42)
    
train_df = pd.DataFrame(train_data)
train_df.to_csv(train_csv, index=False)

test_df = pd.DataFrame(test_data)
test_df.to_csv(test_csv, index=False)
print(cnt)

def compute_channel_statistics(train_data):
    # Translated comment
    num_channels = 12
    channel_sums = np.zeros(num_channels)
    channel_squared_sums = np.zeros(num_channels)
    channel_counts = np.zeros(num_channels)
    
    # Translated comment
    for line in train_data:
        file_path = line['s2_pre_path']
        file_path2 = line['s2_path']
        file_path3 = line['s2_post_path']

        data_s2 = tifffile.imread(file_path)
        for i in range(data_s2.shape[0]):
            data = data_s2[i]
            channel_sums[i] += np.sum(data)
            channel_squared_sums[i] += np.sum(data**2)
            channel_counts[i] += data.size
        data_s2 = tifffile.imread(file_path2)
        for i in range(data_s2.shape[0]):
            data = data_s2[i]
            channel_sums[i] += np.sum(data)
            channel_squared_sums[i] += np.sum(data**2)
            channel_counts[i] += data.size
        data_s2 = tifffile.imread(file_path3)
        for i in range(data_s2.shape[0]):
            data = data_s2[i]
            channel_sums[i] += np.sum(data)
            channel_squared_sums[i] += np.sum(data**2)
            channel_counts[i] += data.size
    # Translated comment
    channel_means = channel_sums / channel_counts
    channel_variances = (channel_squared_sums / channel_counts) - (channel_means**2)
    channel_stds = np.sqrt(channel_variances)
    
    return channel_means, channel_stds
channel_means, channel_stds = compute_channel_statistics(train_data)
channel_means_list = channel_means.tolist()
channel_stds_list = channel_stds.tolist()
print(json.dumps(channel_means_list))
print(json.dumps(channel_stds_list))

total file count 1218
1218
[2085.143706488169, 2301.743936273678, 2715.4353853247903, 3191.4465837615717, 3577.444865403642, 3970.7478262478976, 4190.800817625397, 4389.0916726205705, 0.0, 0.0, 4887.428038864084, 4247.2296062158775]
[1346.8124730478326, 1258.6188390006614, 1208.4573746585163, 1342.0565894487725, 1359.519142531949, 1213.8159528424333, 1206.2444621865147, 1177.2247769977484, 0.0, 0.0, 1385.845010115797, 1403.4246313661085]


In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import rasterio
import tifffile
import json
import random

base_dir = '/home/hongxuan/methane/methane_emission/carbonmapper_data_classification'
train_csv_path = '/home/hongxuan/methane/methane_emission/carbonmapper_data_classification/train.csv'
test_csv_path = '/home/hongxuan/methane/methane_emission/carbonmapper_data_classification/test.csv'
origin_train_csv = '/home/hongxuan/methane/methane_emission/carbonmapper_data/train.csv'
origin_test_csv = '/home/hongxuan/methane/methane_emission/carbonmapper_data/test.csv'

def get_crop(width = 512, height = 512, center_size = 30, crop_width = 96, crop_height = 96):
    
    center_x = width // 2
    center_y = height // 2
    center_left = center_x - center_size // 2
    center_right = center_x + center_size // 2
    center_top = center_y - center_size // 2
    center_bottom = center_y + center_size // 2

    left = random.randint(0, max(0, center_left - crop_width))
    top = random.randint(0, max(0, center_top - crop_height))

    if left + crop_width < center_right:
        left = center_right - crop_width
    if top + crop_height < center_bottom:
        top = center_bottom - crop_height
    return left, top


data = []

image_size = 128

cnt = 0
org_train_df = pd.read_csv(origin_train_csv)
# crop_list = [(0, 0), (0, 512 - 96), (512 - 96, 512 - 96), (512 - 96, 0), (256 - 96, 256 - 96), (256 - 96, 256), (256, 256), (256, 256 - 96)]
for index, row in org_train_df.iterrows():
    t_data = tifffile.imread(row['s2_path'])
    t1_data = tifffile.imread(row['s2_pre_path'])
    t2_data = tifffile.imread(row['s2_post_path'])
    mask = tifffile.imread(row['plume_mask_path'])
    crop_list = []
    for i in range(8):
        crop_list.append(get_crop())
    for i in range(16):
        crop_list.append((random.randint(128, 374), random.randint(128, 374)))
    for crop in crop_list:
        nt_data = t_data[:, crop[0]:crop[0]+image_size,crop[1]:crop[1]+image_size]
        nt1_data = t1_data[:, crop[0]:crop[0]+image_size,crop[1]:crop[1]+image_size]
        nt2_data = t2_data[:, crop[0]:crop[0]+image_size,crop[1]:crop[1]+image_size]
        n_mask = mask[crop[0]:crop[0]+image_size, crop[1]:crop[1]+image_size]
        dir_path = os.path.join(base_dir, str(cnt))
        os.makedirs(dir_path, exist_ok=True)
        nt_path = os.path.join(base_dir, str(cnt), "s2.tif")
        tifffile.imwrite(nt_path, nt_data)
        nt1_path = os.path.join(base_dir, str(cnt), "s2_-5.tif")
        tifffile.imwrite(nt1_path, nt1_data)
        nt2_path = os.path.join(base_dir, str(cnt), "s2_+5.tif")
        tifffile.imwrite(nt2_path, nt2_data)
        n_mask_path = os.path.join(base_dir, str(cnt), "plume.tif")
        tifffile.imwrite(n_mask_path, n_mask)
        mask_sum = np.sum(n_mask)
        data.append({"id": cnt, "s2_path": nt_path, "s2_pre_path": nt1_path, "s2_post_path": nt2_path, "plume_mask_path": n_mask_path, "label": 0 if mask_sum == 0 else 1, "emission_auto": 0 if mask_sum == 0 else row['emission_auto'], "emission_uncertainty_auto": 0 if mask_sum == 0 else row['emission_uncertainty_auto']})
        cnt += 1


train_df = pd.DataFrame(data)
train_df.to_csv(train_csv_path, index=False)
print(f'training set count: {cnt}')


data_test = []
org_test_df = pd.read_csv(origin_test_csv)
for index, row in org_test_df.iterrows():
    t_data = tifffile.imread(row['s2_path'])
    t1_data = tifffile.imread(row['s2_pre_path'])
    t2_data = tifffile.imread(row['s2_post_path'])
    mask = tifffile.imread(row['plume_mask_path'])
    crop_list = []
    for i in range(8):
        crop_list.append(get_crop())

    for i in range(16):
        crop_list.append((random.randint(128, 374), random.randint(128, 374)))
    for crop in crop_list:
        nt_data = t_data[:, crop[0]:crop[0]+image_size,crop[1]:crop[1]+image_size]
        nt1_data = t1_data[:, crop[0]:crop[0]+image_size,crop[1]:crop[1]+image_size]
        nt2_data = t2_data[:, crop[0]:crop[0]+image_size,crop[1]:crop[1]+image_size]
        n_mask = mask[crop[0]:crop[0]+image_size, crop[1]:crop[1]+image_size]
        dir_path = os.path.join(base_dir, str(cnt))
        os.makedirs(dir_path, exist_ok=True)
        nt_path = os.path.join(base_dir, str(cnt), "s2.tif")
        tifffile.imwrite(nt_path, nt_data)
        nt1_path = os.path.join(base_dir, str(cnt), "s2_-5.tif")
        tifffile.imwrite(nt1_path, nt1_data)
        nt2_path = os.path.join(base_dir, str(cnt), "s2_+5.tif")
        tifffile.imwrite(nt2_path, nt2_data)
        n_mask_path = os.path.join(base_dir, str(cnt), "plume.tif")
        tifffile.imwrite(n_mask_path, n_mask)
        mask_sum = np.sum(n_mask)
        data_test.append({"id": cnt, "s2_path": nt_path, "s2_pre_path": nt1_path, "s2_post_path": nt2_path, "plume_mask_path": n_mask_path, "label": 0 if mask_sum == 0 else 1, "emission_auto": 0 if mask_sum == 0 else row['emission_auto'], "emission_uncertainty_auto": 0 if mask_sum == 0 else row['emission_uncertainty_auto']})
        cnt += 1

test_df = pd.DataFrame(data_test)
test_df.to_csv(test_csv_path, index=False)
print(f'total count: {cnt}')

column_sum = train_df['label'].sum()
print(f"label: {column_sum} total {len(train_df)}")

column_sum = test_df['label'].sum()
print(f"label: {column_sum} total {len(test_df)}")

training set count: 23376
total count: 29232
